# 01 - Data Preparation: Multi-Component RUL (FD004)
In this notebook, we will:
- Load C-MAPSS FD004 dataset
- Simulate 3 component groups based on sensor subsets
- Generate RUL labels for each component
- Prepare time-series sequences using sliding windows

In [13]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

import sys
import os
sys.path.append(os.path.abspath("../src"))
from data_utils import generate_rul_targets, create_sequence_dataset

In [14]:
# Load FD004 dataset from NASA C-MAPSS
cols = ['unit', 'cycle', 'setting1', 'setting2', 'setting3'] + [f'sensor{i}' for i in range(1, 22)]
df = pd.read_csv('../data/CMAPSSData/train_FD004.txt', sep='\s+', header=None)
df.columns = cols
df.head()

<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
/var/folders/7n/jsxq10792314ly6yrv8s3vn80000gn/T/ipykernel_45560/2935636931.py:3: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv('../data/CMAPSSData/train_FD004.txt', sep='\s+', header=None)


,unit,cycle,setting1,setting2,setting3,sensor1,sensor2,sensor3,sensor4,sensor5,...,sensor12,sensor13,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,129.78,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,312.59,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,129.62,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,129.80,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,164.11,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754


In [15]:
# Simulate 3 components using sensor groups
component_map = {
    'comp1': [f'sensor{i}' for i in range(2, 8)],
    'comp2': [f'sensor{i}' for i in range(8, 15)],
    'comp3': [f'sensor{i}' for i in range(15, 22)]
}
components = list(component_map.keys())

In [16]:
# Add RUL columns for each simulated component
rul_df = generate_rul_targets(df, components=components)
rul_df.head()

,unit,cycle,setting1,setting2,setting3,sensor1,sensor2,sensor3,sensor4,sensor5,...,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21,RUL_comp1,RUL_comp2,RUL_comp3
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,9.3335,0.02,330,2212,100.00,10.62,6.3670,320,320,320
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,9.1913,0.02,361,2324,100.00,24.37,14.6552,319,319,319
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,9.4007,0.02,329,2212,100.00,10.48,6.4213,318,318,318
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,9.3369,0.02,328,2212,100.00,10.54,6.4176,317,317,317
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,10.8366,0.02,305,1915,84.93,14.03,8.6754,316,316,316


In [17]:
# Normalize sensor columns
sensor_cols = [col for col in df.columns if 'sensor' in col]
scaler = MinMaxScaler()
df[sensor_cols] = scaler.fit_transform(df[sensor_cols])

In [20]:
# Create time-series sequences and multi-output RUL targets
X, y = create_sequence_dataset(df, rul_df, window_size=30, components=components)

import os
os.makedirs('outputs', exist_ok=True)

# Save processed arrays
np.save('../outputs/X.npy', X)
np.save('../outputs/y.npy', y)

print(f'Shape of input sequences: {X.shape}')
print(f'Shape of RUL targets: {y.shape}')

Shape of input sequences: (53779, 30, 21)
Shape of RUL targets: (53779, 3)
